# 01 — Quickstart

Load `compliance-v1`, run inference in all three modes, and understand uncertainty decomposition.

**Time:** ~5 minutes

In [ ]:
# Cell 0 — Environment setup (Colab / JupyterHub compatible)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "epalea", "-q"])
import epalea
print(f"epalea {epalea.__version__} ready")

## Load the compliance-v1 model

This loads `best_model.pt`, `aggregator.pt`, and the pre-built FAISS evidence index.

In [ ]:
import epalea

# List available pretrained models
models = epalea.list_models()
for m in models:
    print(f"  {m['model_id']:20} — {m['description']}")

In [ ]:
# Load compliance-v1
model = epalea.load_model("compliance-v1")
print(f"Available modes: {model.available_modes}")

## Single inference — SPN mode (default)

The SPN path is the primary research contribution. Works without the aggregator.

In [ ]:
# Infer for a single entity using SPN
result = model.infer(entity_id="C0001", mode="spn")

print(f"Entity:     {result.entity_id}")
print(f"Mode:       {result.mode}")
print(f"Prediction: {result.results.spn.prediction}")
print(f"Confidence: {result.results.spn.confidence:.3f}")
print(f"Distribution: {result.results.spn.distribution}")

## Both modes — side-by-side comparison

Runs SPN and Aggregator paths and returns results + uncertainty **separately for each mode**.
Neither uncertainty signal is suppressed when `mode="both"`.


In [ ]:
# Run both modes at once
result = model.infer(entity_id="C0001", mode="both")

print("SPN:")
print(f"  Prediction:  {result.results.spn.prediction}")
print(f"  Confidence:  {result.results.spn.confidence:.3f}")
print(f"  Epistemic:   {result.uncertainty.spn.epistemic:.3f}")
print(f"  Aleatoric:   {result.uncertainty.spn.aleatoric:.3f}")
print(f"  Total:       {result.uncertainty.spn.total:.3f}")

print("\nAggregator:")
print(f"  Prediction:  {result.results.aggregator.prediction}")
print(f"  Confidence:  {result.results.aggregator.confidence:.3f}")
print(f"  Epistemic:   {result.uncertainty.aggregator.epistemic:.3f}")
print(f"  Aleatoric:   {result.uncertainty.aggregator.aleatoric:.3f}")
print(f"  Total:       {result.uncertainty.aggregator.total:.3f}")

# Verify neither is suppressed
assert result.uncertainty.spn is not None, "FAIL: uncertainty.spn is None"
assert result.uncertainty.aggregator is not None, "FAIL: uncertainty.aggregator is None"
print("\n✓ Both uncertainty decompositions present")


## Flat output format

For downstream processing, use `output_format='flat'` to get all fields at the top level.

In [ ]:
import json

result_flat = model.infer(entity_id="C0001", mode="both", output_format="flat")
print(json.dumps(result_flat.to_dict(), indent=2))

## Understanding `weights_source`

- `'uniform'` — SPN path uses equal weights across all evidence items
- `'aggregator'` — Learned path uses trained quality/consistency weights

When `mode="both"`, uncertainty is always reported per-mode:

```python
result.uncertainty.spn.weights_source        # "uniform"
result.uncertainty.aggregator.weights_source # "aggregator"
```

The two signals are computed independently — comparing them tells you how much the
structured (SPN) and learned (Aggregator) paths agree on uncertainty.


In [ ]:
# The raw dict is always accessible
print(json.dumps(result.to_dict(), indent=2))

## What's next

- **Notebook 02** — Train your own LPF model on the compliance data
- **Notebook 03** — Batch inference, evaluation metrics, calibration curves
- **Notebook 04** — Define a custom domain and train a model from scratch